In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# DOWNLOAD STOCK DATA USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'SPY'  # Best on indices/ETFs with volume (SPY, QQQ, DIA, IWM)
START_DATE = '2018-01-01'

stock_data = yf.download(TICKER, start=START_DATE, interval='1d')

if not stock_data.empty:
    print(f'Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}')
    print(f'Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}')
    print('\nFirst 5 rows:')
    print(stock_data.head())
else:
    print(f'Failed to download {TICKER} data from yfinance')

In [ ]:
# TECHNICAL INDICATORS

# Handle MultiIndex columns from yfinance
if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[('Close', TICKER)].astype(float)
    high = stock_data[('High', TICKER)].astype(float)
    low = stock_data[('Low', TICKER)].astype(float)
    open_ = stock_data[('Open', TICKER)].astype(float)
    volume = stock_data[('Volume', TICKER)].astype(float)
else:
    close = stock_data['Close'].astype(float)
    high = stock_data['High'].astype(float)
    low = stock_data['Low'].astype(float)
    open_ = stock_data['Open'].astype(float)
    volume = stock_data['Volume'].astype(float)

# Standard indicator suite
SMA_20 = pd.Series(talib.SMA(close.values, timeperiod=20), index=close.index, name='SMA_20')
SMA_50 = pd.Series(talib.SMA(close.values, timeperiod=50), index=close.index, name='SMA_50')
EMA_12 = pd.Series(talib.EMA(close.values, timeperiod=12), index=close.index, name='EMA_12')
EMA_26 = pd.Series(talib.EMA(close.values, timeperiod=26), index=close.index, name='EMA_26')
macd_line, macd_signal, macd_hist = talib.MACD(close.values)
RSI_14 = pd.Series(talib.RSI(close.values, timeperiod=14), index=close.index, name='RSI_14')

# Rolling VWAP (core indicator for this strategy)
typical_price = (high + low + close) / 3
for vp in [10, 20, 30]:
    rv = (typical_price * volume).rolling(vp).sum() / volume.rolling(vp).sum()
    print(f'Rolling VWAP({vp}) latest: {rv.iloc[-1]:.2f}')

indicators_df = pd.DataFrame({
    'Close': close, 'High': high, 'Low': low, 'Volume': volume,
    'SMA_20': SMA_20, 'SMA_50': SMA_50,
    'EMA_12': EMA_12, 'EMA_26': EMA_26,
    'RSI_14': RSI_14,
    'Typical_Price': typical_price,
})
print('\nIndicators DataFrame — last 5 rows:')
print(indicators_df.tail(5))

In [ ]:
# PREPARE PRICE SERIES + TRAIN / VAL SPLIT

def select_close_series(data, ticker):
    """Extract close series handling MultiIndex."""
    if isinstance(data.columns, pd.MultiIndex):
        return data[('Close', ticker)].astype(float)
    return data['Close'].astype(float)

TRAIN_RATIO = 0.60
split_idx = int(len(close) * TRAIN_RATIO)

train_close = close.iloc[:split_idx]
train_high = high.iloc[:split_idx]
train_low = low.iloc[:split_idx]
train_volume = volume.iloc[:split_idx]

val_close = close.iloc[split_idx:]
val_high = high.iloc[split_idx:]
val_low = low.iloc[split_idx:]
val_volume = volume.iloc[split_idx:]

print(f'Train: {train_close.index[0].date()} to {train_close.index[-1].date()} ({len(train_close)} bars)')
print(f'Val:   {val_close.index[0].date()} to {val_close.index[-1].date()} ({len(val_close)} bars)')
print(f'Split ratio: {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')

## Brian Shannon — VWAP Bounce Strategy

Based on Brian Shannon's *Technical Analysis Using Multiple Timeframes* — the VWAP acts as an
institutional-grade dynamic support/resistance level. When price is in an uptrend and pulls back
to VWAP, institutional buyers step in, creating a high-probability bounce.

### Adapted for Daily Timeframe
We use a **rolling VWAP** (volume-weighted average price over N days) instead of intraday reset VWAP.

### Strategy Logic (Long Only)
1. Compute Rolling VWAP over `vwap_period` days
2. **Trend Up**: VWAP is sloping upward (VWAP today > VWAP `slope_lookback` bars ago)
3. **Pullback Zone**: Price pulls back to within `pullback_pct` below VWAP
4. **Bounce Entry**: Price was in pullback zone on previous bar, now closes above VWAP → BUY
5. **Exit**: Close < VWAP (trend/support broken)

All signals shifted by 1 bar for execution delay (no lookahead bias).

### Grid Search Parameters
- `vwap_period`: [10, 15, 20, 25, 30, 40] — rolling window for VWAP calculation
- `pullback_pct`: [0.005, 0.01, 0.015, 0.02, 0.025] — how close price must get to VWAP (0.5%–2.5%)
- `slope_lookback`: [3, 5, 8, 10, 15] — bars to measure VWAP slope direction

In [ ]:
# DEFINE PARAMETER RANGES

vwap_period_range = [10, 15, 20, 25, 30, 40]
pullback_pct_range = [0.005, 0.01, 0.015, 0.02, 0.025]
slope_lookback_range = [3, 5, 8, 10, 15]

total_combos = len(vwap_period_range) * len(pullback_pct_range) * len(slope_lookback_range)

print('Parameter Ranges:')
print(f'  vwap_period:     {vwap_period_range}')
print(f'  pullback_pct:    {pullback_pct_range}')
print(f'  slope_lookback:  {slope_lookback_range}')
print(f'\nTotal combinations: {total_combos}')

# Preview first 10
from itertools import islice, product
print('\nFirst 10 combinations preview:')
for i, (vp, pp, sl) in enumerate(islice(product(vwap_period_range, pullback_pct_range, slope_lookback_range), 10)):
    print(f'  {i+1}. vwap_period={vp}, pullback_pct={pp}, slope_lookback={sl}')

In [ ]:
# INITIALIZE RESULTS COLLECTION

grid_search_results = []

# Full metric list (31 metrics)
METRIC_LIST = [
    'vwap_period', 'pullback_pct', 'slope_lookback',
    'total_return', 'annualized_return',
    'sharpe_ratio', 'sortino_ratio', 'calmar_ratio', 'omega_ratio',
    'information_ratio', 'tail_ratio', 'deflated_sharpe_ratio',
    'max_drawdown', 'volatility', 'ulcer_index',
    'win_rate', 'total_trades', 'avg_trade_duration',
    'expectancy', 'profit_factor', 'sqn',
    'payoff_ratio', 'largest_win', 'largest_loss',
    'avg_win_amount', 'avg_loss_amount',
    'winning_streak', 'losing_streak',
    'recovery_factor', 'gain_to_pain_ratio', 'serenity_index',
    'trades_per_year'
]
print(f'Tracking {len(METRIC_LIST)} metrics per combination:')
for i, m in enumerate(METRIC_LIST):
    print(f'  {i+1:2d}. {m}')

In [ ]:
# GRID SEARCH — TRAINING SET ONLY

INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
MIN_TRADES = 10

def compute_rolling_vwap(close_s, high_s, low_s, volume_s, period):
    """Compute rolling VWAP over N bars."""
    tp = (high_s + low_s + close_s) / 3
    rolling_vwap = (tp * volume_s).rolling(period).sum() / volume_s.rolling(period).sum()
    return rolling_vwap


def generate_vwap_bounce_signals(close_s, high_s, low_s, volume_s,
                                  vwap_period, pullback_pct, slope_lookback):
    """
    Brian Shannon VWAP Bounce: long when price pulls back to VWAP and bounces in uptrend.
    Returns: entries, exits (both pd.Series[bool])
    """
    rv = compute_rolling_vwap(close_s, high_s, low_s, volume_s, vwap_period)

    # Trend filter: VWAP sloping up
    vwap_slope_up = rv > rv.shift(slope_lookback)

    # Pullback zone: price is within pullback_pct below VWAP (or touching it)
    lower_bound = rv * (1 - pullback_pct)
    pullback_zone = (close_s >= lower_bound) & (close_s <= rv)

    # Bounce: was in pullback zone last bar, now above VWAP, trend is up
    entries_raw = pullback_zone.shift(1) & (close_s > rv) & vwap_slope_up

    # Exit: close below VWAP
    exits_raw = close_s < rv

    # Apply 1-bar execution delay
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)

    return entries, exits


def safe_metric(fn, default=np.nan):
    try:
        val = fn()
        return float(val) if val is not None and not np.isinf(val) else default
    except:
        return default


def extract_full_metrics(pf, n_years):
    """Extract all 31 metrics from a vectorbt portfolio."""
    trades = pf.trades
    tr = np.asarray(trades.returns.values if hasattr(trades.returns, 'values') else trades.returns).ravel()
    pnl = np.asarray(trades.pnl.values if hasattr(trades.pnl, 'values') else trades.pnl).ravel()
    pos = tr[tr > 0]
    neg = tr[tr < 0]

    dd = safe_metric(pf.max_drawdown)
    ann_ret = safe_metric(lambda: pf.annualized_return(freq='D'))

    return {
        'total_return': safe_metric(pf.total_return),
        'annualized_return': ann_ret,
        'sharpe_ratio': safe_metric(lambda: pf.sharpe_ratio(freq='D')),
        'sortino_ratio': safe_metric(lambda: pf.sortino_ratio(freq='D')),
        'calmar_ratio': ann_ret / abs(dd) if dd and abs(dd) > 1e-9 and not np.isnan(ann_ret) else np.nan,
        'omega_ratio': np.nan,
        'information_ratio': np.nan,
        'tail_ratio': np.nan,
        'deflated_sharpe_ratio': np.nan,
        'max_drawdown': dd,
        'volatility': safe_metric(lambda: pf.annualized_volatility(freq='D')),
        'ulcer_index': np.nan,
        'win_rate': float(len(pos) / len(tr) * 100) if len(tr) > 0 else np.nan,
        'total_trades': len(trades),
        'avg_trade_duration': np.nan,
        'expectancy': float(tr.mean()) if len(tr) > 0 else np.nan,
        'profit_factor': float(pos.sum() / abs(neg.sum())) if len(neg) > 0 and abs(neg.sum()) > 0 else np.nan,
        'sqn': float(tr.mean() / tr.std() * np.sqrt(len(tr))) if len(tr) > 1 and tr.std() > 0 else np.nan,
        'payoff_ratio': float(abs(pos.mean() / neg.mean())) if len(pos) > 0 and len(neg) > 0 else np.nan,
        'largest_win': float(pos.max()) if len(pos) > 0 else np.nan,
        'largest_loss': float(neg.min()) if len(neg) > 0 else np.nan,
        'avg_win_amount': float(pos.mean()) if len(pos) > 0 else np.nan,
        'avg_loss_amount': float(neg.mean()) if len(neg) > 0 else np.nan,
        'winning_streak': np.nan,
        'losing_streak': np.nan,
        'recovery_factor': np.nan,
        'gain_to_pain_ratio': np.nan,
        'serenity_index': np.nan,
        'trades_per_year': len(trades) / max(n_years, 1e-9),
    }


# ---- RUN GRID SEARCH ----
combo_count = 0
train_years = max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9)

for vwap_period in vwap_period_range:
    for pullback_pct in pullback_pct_range:
        for slope_lookback in slope_lookback_range:
            combo_count += 1

            try:
                entries, exits = generate_vwap_bounce_signals(
                    train_close, train_high, train_low, train_volume,
                    vwap_period=vwap_period,
                    pullback_pct=pullback_pct,
                    slope_lookback=slope_lookback
                )

                if entries.sum() < 2:
                    continue

                pf = vbt.Portfolio.from_signals(
                    train_close, entries=entries, exits=exits,
                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
                )

                m = extract_full_metrics(pf, train_years)

                if m['total_trades'] < MIN_TRADES:
                    continue

                grid_search_results.append({
                    'vwap_period': vwap_period,
                    'pullback_pct': pullback_pct,
                    'slope_lookback': slope_lookback,
                    **m
                })
            except Exception as e:
                pass

            if combo_count % 50 == 0:
                print(f'\U0001f504 Completed {combo_count}/{total_combos} combinations...')

print(f'\nCompleted all {combo_count} combinations. Valid results: {len(grid_search_results)}')

# Convert to DataFrame and show top 10
results_df = pd.DataFrame(grid_search_results)

if len(results_df) > 0:
    top10 = results_df.nlargest(10, 'sharpe_ratio')
    print(f'\nTop 10 by IS Sharpe:')
    print('=' * 120)
    print(f'{"Rank":<5} {"VWAP_P":<8} {"PB_Pct":<8} {"Slope":<7} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Trades":>7} {"WinRate":>8} {"PF":>8}')
    print('-' * 120)
    for i, (_, r) in enumerate(top10.iterrows()):
        s = f"{r['sharpe_ratio']:.3f}" if pd.notna(r['sharpe_ratio']) else 'N/A'
        ret = f"{r['total_return']:.2%}" if pd.notna(r['total_return']) else 'N/A'
        dd = f"{r['max_drawdown']:.2%}" if pd.notna(r['max_drawdown']) else 'N/A'
        wr = f"{r['win_rate']:.1f}%" if pd.notna(r['win_rate']) else 'N/A'
        pf_val = f"{r['profit_factor']:.2f}" if pd.notna(r['profit_factor']) else 'N/A'
        print(f"{i+1:<5} {r['vwap_period']:<8} {r['pullback_pct']:<8.3f} {r['slope_lookback']:<7} {s:>8} {ret:>10} {dd:>10} {int(r['total_trades']):>7} {wr:>8} {pf_val:>8}")
else:
    print('No valid results found.')

In [ ]:
# VALIDATE TOP 5 ON OOS DATA

if len(results_df) == 0:
    print('No valid results to validate.')
else:
    top_5 = results_df.nlargest(5, 'sharpe_ratio')
    val_years = max((val_close.index[-1] - val_close.index[0]).days / 365.25, 1e-9)

    print('Top 5 IS performers — OOS Validation')
    print('=' * 140)
    print(f'{"Rank":<5} {"VWAP_P":<8} {"PB_Pct":<8} {"Slope":<7} {"IS Sharpe":>10} {"OOS Sharpe":>11} {"IS Return":>10} {"OOS Return":>11} {"IS MaxDD":>10} {"OOS MaxDD":>11} {"OOS Trades":>10}')
    print('-' * 140)

    fig, axes = plt.subplots(1, min(len(top_5), 5), figsize=(20, 5), squeeze=False)

    for i, (_, r) in enumerate(top_5.iterrows()):
        ent_val, ext_val = generate_vwap_bounce_signals(
            val_close, val_high, val_low, val_volume,
            vwap_period=int(r['vwap_period']),
            pullback_pct=r['pullback_pct'],
            slope_lookback=int(r['slope_lookback'])
        )

        pf_val = vbt.Portfolio.from_signals(
            val_close, entries=ent_val, exits=ext_val,
            init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
        )
        m_val = extract_full_metrics(pf_val, val_years)

        is_s = f"{r['sharpe_ratio']:.3f}" if pd.notna(r['sharpe_ratio']) else 'N/A'
        oos_s = f"{m_val['sharpe_ratio']:.3f}" if pd.notna(m_val['sharpe_ratio']) else 'N/A'
        is_r = f"{r['total_return']:.2%}" if pd.notna(r['total_return']) else 'N/A'
        oos_r = f"{m_val['total_return']:.2%}" if pd.notna(m_val['total_return']) else 'N/A'
        is_d = f"{r['max_drawdown']:.2%}" if pd.notna(r['max_drawdown']) else 'N/A'
        oos_d = f"{m_val['max_drawdown']:.2%}" if pd.notna(m_val['max_drawdown']) else 'N/A'

        print(f"{i+1:<5} {int(r['vwap_period']):<8} {r['pullback_pct']:<8.3f} {int(r['slope_lookback']):<7} {is_s:>10} {oos_s:>11} {is_r:>10} {oos_r:>11} {is_d:>10} {oos_d:>11} {int(m_val['total_trades']):>10}")

        # Plot equity curve
        equity = pf_val.value()
        ax = axes[0][i]
        ax.plot(equity.index, equity.values, linewidth=1.5)
        ax.set_title(f'#{i+1} vp={int(r["vwap_period"])} pb={r["pullback_pct"]:.3f}\nOOS Sharpe: {oos_s}', fontsize=9)
        ax.set_ylabel('Portfolio Value ($)')
        ax.tick_params(axis='x', rotation=45, labelsize=7)

    plt.tight_layout()
    plt.show()

## Sensitivity Analysis

For each parameter (`vwap_period`, `pullback_pct`, `slope_lookback`), we vary it while holding
the others fixed at their best values.

**Color coding**: Dark green (>+10% vs base), Light green (0–10%), Orange (-10–0%), Red (<-10%).

In [ ]:
# SENSITIVITY ANALYSIS

if len(results_df) == 0:
    print('No results for sensitivity analysis.')
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    vwap_base = int(best['vwap_period'])
    pb_base = best['pullback_pct']
    slope_base = int(best['slope_lookback'])
    base_sharpe = best['sharpe_ratio']

    print(f'Base combo: vwap_period={vwap_base}, pullback_pct={pb_base}, slope_lookback={slope_base}')
    print(f'Base IS Sharpe: {base_sharpe:.3f}')

    param_configs = [
        ('vwap_period', vwap_period_range, vwap_base),
        ('pullback_pct', pullback_pct_range, pb_base),
        ('slope_lookback', slope_lookback_range, slope_base),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    sensitivity_summary = []

    for col, (param_name, param_values, base_val) in enumerate(param_configs):
        is_sharpes = []
        oos_sharpes = []

        for val in param_values:
            kwargs = {
                'vwap_period': vwap_base,
                'pullback_pct': pb_base,
                'slope_lookback': slope_base,
            }
            kwargs[param_name] = val

            # IS
            try:
                ent, ext = generate_vwap_bounce_signals(train_close, train_high, train_low, train_volume, **kwargs)
                if ent.sum() >= 2:
                    pf_is = vbt.Portfolio.from_signals(train_close, entries=ent, exits=ext,
                                                       init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
                    is_sharpes.append(safe_metric(lambda: pf_is.sharpe_ratio(freq='D')))
                else:
                    is_sharpes.append(np.nan)
            except:
                is_sharpes.append(np.nan)

            # OOS
            try:
                ent_v, ext_v = generate_vwap_bounce_signals(val_close, val_high, val_low, val_volume, **kwargs)
                if ent_v.sum() >= 2:
                    pf_oos = vbt.Portfolio.from_signals(val_close, entries=ent_v, exits=ext_v,
                                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
                    oos_sharpes.append(safe_metric(lambda: pf_oos.sharpe_ratio(freq='D')))
                else:
                    oos_sharpes.append(np.nan)
            except:
                oos_sharpes.append(np.nan)

        for row, (sharpes, label) in enumerate([(is_sharpes, 'IS'), (oos_sharpes, 'OOS')]):
            ax = axes[row][col]
            colors = []
            for s in sharpes:
                if pd.isna(s) or base_sharpe == 0:
                    colors.append('gray')
                else:
                    pct = (s - base_sharpe) / abs(base_sharpe) if base_sharpe != 0 else 0
                    if pct > 0.10: colors.append('#006400')
                    elif pct >= 0: colors.append('#90EE90')
                    elif pct >= -0.10: colors.append('#FFA500')
                    else: colors.append('#DC143C')

            x_labels = [str(v) for v in param_values]
            vals_plot = [s if pd.notna(s) else 0 for s in sharpes]
            ax.bar(x_labels, vals_plot, color=colors, alpha=0.8)

            if base_val in param_values:
                base_x = param_values.index(base_val)
                ax.axvline(x=base_x, color='blue', linestyle='--', alpha=0.7, linewidth=2)

            ax.set_title(f'{label} Sharpe vs {param_name}')
            ax.set_ylabel('Sharpe')
            ax.axhline(y=0, color='black', linewidth=0.5)

        # Sensitivity flag
        valid_is = [s for s in is_sharpes if pd.notna(s)]
        if len(valid_is) > 1 and base_sharpe != 0:
            spread = (max(valid_is) - min(valid_is)) / abs(base_sharpe)
            flag = 'LOW' if spread < 0.30 else 'HIGH'
            sensitivity_summary.append((param_name, spread, flag))

    plt.tight_layout()
    plt.show()

    print('\nSENSITIVITY SUMMARY')
    print('=' * 60)
    for name, spread, flag in sensitivity_summary:
        icon = '\u2705 LOW' if flag == 'LOW' else '\u26a0\ufe0f HIGH'
        print(f'  {name:20s}  spread={spread:.1%}  [{icon}]')

In [ ]:
# FULL-SAMPLE EVALUATION + BUY & HOLD COMPARISON

if len(results_df) == 0:
    print('No results to evaluate.')
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    best_params = {
        'vwap_period': int(best['vwap_period']),
        'pullback_pct': best['pullback_pct'],
        'slope_lookback': int(best['slope_lookback']),
    }

    # Full-sample backtest
    ent_full, ext_full = generate_vwap_bounce_signals(
        close, high, low, volume, **best_params
    )
    pf_full = vbt.Portfolio.from_signals(
        close, entries=ent_full, exits=ext_full,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
    )
    full_years = max((close.index[-1] - close.index[0]).days / 365.25, 1e-9)
    m_full = extract_full_metrics(pf_full, full_years)

    # Buy and hold
    bh_e = pd.Series(False, index=close.index, dtype=bool); bh_e.iloc[0] = True
    bh_x = pd.Series(False, index=close.index, dtype=bool)
    pf_bh = vbt.Portfolio.from_signals(close, entries=bh_e, exits=bh_x,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')
    m_bh = extract_full_metrics(pf_bh, full_years)

    param_str = ', '.join(f'{k}={v}' for k, v in best_params.items())
    print(f'Best Combo: {param_str}')
    print()
    print(f'{"Metric":<20} {"Strategy":>12} {"Buy & Hold":>12}')
    print('-' * 50)
    print(f'{"Sharpe":<20} {m_full["sharpe_ratio"]:>12.3f} {m_bh["sharpe_ratio"]:>12.3f}')
    print(f'{"Total Return":<20} {m_full["total_return"]:>12.2%} {m_bh["total_return"]:>12.2%}')
    print(f'{"Max Drawdown":<20} {m_full["max_drawdown"]:>12.2%} {m_bh["max_drawdown"]:>12.2%}')
    print(f'{"Trades":<20} {int(m_full["total_trades"]):>12d} {"N/A":>12}')
    print(f'{"Win Rate":<20} {m_full["win_rate"]:>11.1f}% {"N/A":>12}')
    print(f'{"Profit Factor":<20} {m_full["profit_factor"]:>12.2f} {"N/A":>12}') if pd.notna(m_full['profit_factor']) else None

    # Equity curve + drawdown
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

    equity_strat = pf_full.value()
    equity_bh = pf_bh.value()

    ax1.plot(close.index[:split_idx], equity_strat.iloc[:split_idx].values,
             color='#3498db', linewidth=1.5, label='VWAP Bounce (IS)')
    ax1.plot(close.index[split_idx:], equity_strat.iloc[split_idx:].values,
             color='#e67e22', linewidth=1.5, label='VWAP Bounce (OOS)')
    ax1.plot(close.index, equity_bh.values, label='Buy & Hold',
             linewidth=1.5, color='gray', alpha=0.7, linestyle='--')

    split_date = close.index[split_idx]
    ax1.axvline(x=split_date, color='red', linestyle='--', alpha=0.7, label='Train/Val Split')

    stats_text = (f'Sharpe: {m_full["sharpe_ratio"]:.3f}\n'
                  f'Return: {m_full["total_return"]:.2%}\n'
                  f'MaxDD: {m_full["max_drawdown"]:.2%}\n'
                  f'Trades: {int(m_full["total_trades"])}\n'
                  f'WinRate: {m_full["win_rate"]:.1f}%')
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, fontsize=9,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax1.set_title(f'VWAP Bounce Strategy — {TICKER} Full Sample')
    ax1.set_ylabel('Portfolio Value ($)')
    ax1.legend(loc='upper left', bbox_to_anchor=(0.15, 1.0))

    dd = pf_full.drawdown()
    ax2.fill_between(dd.index, dd.values, 0, color='red', alpha=0.3)
    ax2.plot(dd.index, dd.values, color='red', linewidth=0.8)
    ax2.set_title('Drawdown')
    ax2.set_ylabel('Drawdown')
    ax2.set_xlabel('Date')

    plt.tight_layout()
    plt.show()

In [ ]:
# TRADE-BY-TRADE P&L ANALYSIS

if len(results_df) == 0:
    print('No results.')
else:
    trades_obj = pf_full.trades
    tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
    pnl = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()

    if len(tr) > 0:
        n = len(tr)
        colors_bar = ['#2ca02c' if r > 0 else '#e74c3c' for r in tr]
        colors_pnl = ['#2ca02c' if p > 0 else '#e74c3c' for p in pnl]

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # 1. Trade returns bar chart
        axes[0][0].bar(range(n), tr * 100, color=colors_bar, edgecolor='none', width=0.8)
        axes[0][0].axhline(np.mean(tr) * 100, color='#3498db', linestyle='--', linewidth=1.5,
                           label=f'Mean: {np.mean(tr)*100:.2f}%')
        axes[0][0].set_title('Trade Returns (%)')
        axes[0][0].set_xlabel('Trade #')
        axes[0][0].legend()

        # 2. Trade P&L bar chart
        axes[0][1].bar(range(n), pnl, color=colors_pnl, edgecolor='none', width=0.8)
        axes[0][1].axhline(np.mean(pnl), color='#3498db', linestyle='--', linewidth=1.5)
        axes[0][1].set_title('Trade P&L ($)')
        axes[0][1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

        # 3. Cumulative P&L
        cum_pnl = np.cumsum(pnl)
        axes[1][0].plot(range(1, n + 1), cum_pnl, color='#3498db', linewidth=2, marker='o', markersize=3)
        axes[1][0].fill_between(range(1, n + 1), cum_pnl, 0, where=cum_pnl >= 0, alpha=0.15, color='green')
        axes[1][0].fill_between(range(1, n + 1), cum_pnl, 0, where=cum_pnl < 0, alpha=0.15, color='red')
        axes[1][0].set_title('Cumulative P&L')
        axes[1][0].set_xlabel('Trade #')
        axes[1][0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

        # 4. Return distribution
        axes[1][1].hist(tr * 100, bins=min(30, max(10, n // 3)), color='steelblue', alpha=0.7, edgecolor='black')
        axes[1][1].axvline(np.mean(tr) * 100, color='#e74c3c', linestyle='--', linewidth=2,
                           label=f'Mean: {np.mean(tr)*100:.2f}%')
        axes[1][1].set_title('Return Distribution')
        axes[1][1].set_xlabel('Return (%)')
        axes[1][1].legend()

        plt.tight_layout()
        plt.show()
    else:
        print('No trades to analyze.')

In [ ]:
# MONTE CARLO FTMO SIMULATION (10,000 paths)

FTMO_ACCOUNT = 100_000
FTMO_PROFIT_TARGET = 0.10
FTMO_DAILY_LOSS_LIMIT = 0.05
FTMO_TOTAL_LOSS_LIMIT = 0.10
FTMO_DAYS = 30
N_SIMULATIONS = 10_000

if len(results_df) == 0:
    print('No results.')
else:
    daily_rets = pf_full.daily_returns().dropna().values

    if len(daily_rets) < 10:
        print('Not enough daily returns for Monte Carlo.')
    else:
        np.random.seed(42)
        pass_count = 0
        fail_daily = 0
        fail_total = 0
        fail_no_target = 0
        final_balances = []
        sample_paths = []

        for sim in range(N_SIMULATIONS):
            sampled = np.random.choice(daily_rets, size=FTMO_DAYS, replace=True)
            balance = FTMO_ACCOUNT
            path = [FTMO_ACCOUNT]
            passed = False
            failed = False
            fail_reason = ''

            for day_ret in sampled:
                daily_pnl = balance * day_ret
                balance += daily_pnl
                path.append(balance)

                if daily_pnl < -FTMO_ACCOUNT * FTMO_DAILY_LOSS_LIMIT:
                    failed = True; fail_reason = 'daily'; break
                if balance < FTMO_ACCOUNT * (1 - FTMO_TOTAL_LOSS_LIMIT):
                    failed = True; fail_reason = 'total'; break
                if balance >= FTMO_ACCOUNT * (1 + FTMO_PROFIT_TARGET):
                    passed = True; break

            while len(path) < FTMO_DAYS + 1:
                path.append(path[-1])
            final_balances.append(balance)
            if sim < 150:
                sample_paths.append(path)

            if passed:
                pass_count += 1
            elif failed and fail_reason == 'daily':
                fail_daily += 1
            elif failed and fail_reason == 'total':
                fail_total += 1
            else:
                fail_no_target += 1

        pass_rate = pass_count / N_SIMULATIONS * 100
        verdict = 'FAVORABLE' if pass_rate >= 50 else 'POSSIBLE' if pass_rate >= 25 else 'CHALLENGING' if pass_rate >= 10 else 'UNLIKELY'

        print(f'FTMO Monte Carlo Simulation ({N_SIMULATIONS:,} paths, {FTMO_DAYS} days each)')
        print('=' * 60)
        print(f'Pass Rate:           {pass_rate:.1f}%')
        print(f'Failed (Daily Loss): {fail_daily/N_SIMULATIONS:.1%}')
        print(f'Failed (Total Loss): {fail_total/N_SIMULATIONS:.1%}')
        print(f'Failed (No Target):  {fail_no_target/N_SIMULATIONS:.1%}')
        print(f'\nMedian Final Balance: ${np.median(final_balances):,.0f}')
        print(f'Mean Final Balance:   ${np.mean(final_balances):,.0f}')
        print(f'\nVerdict: {verdict}')

        # Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # Simulation paths
        for path in sample_paths:
            c = '#2ca02c' if path[-1] >= 110000 else ('#e74c3c' if path[-1] <= 90000 else '#555555')
            ax1.plot(range(FTMO_DAYS + 1), path, color=c, alpha=0.25, linewidth=0.5)
        ax1.axhline(110000, color='#2ca02c', linestyle='--', linewidth=2.5, label='10% Target ($110k)')
        ax1.axhline(90000, color='#e74c3c', linestyle='--', linewidth=2.5, label='10% Max Loss ($90k)')
        ax1.axhline(100000, color='white', linestyle='-', linewidth=0.5, alpha=0.3)
        ax1.set_title(f'FTMO Monte Carlo — Pass Rate: {pass_rate:.1f}% ({verdict})')
        ax1.set_xlabel('Trading Day')
        ax1.set_ylabel('Equity ($)')
        ax1.legend(fontsize=8)

        # Outcome pie
        labels = ['Passed', 'Failed (Daily)', 'Failed (Total)', 'No Target']
        sizes = [pass_count, fail_daily, fail_total, fail_no_target]
        colors_pie = ['#2ecc71', '#e74c3c', '#c0392b', '#f39c12']
        non_zero = [(l, s, c) for l, s, c in zip(labels, sizes, colors_pie) if s > 0]
        if non_zero:
            ax2.pie([x[1] for x in non_zero], labels=[x[0] for x in non_zero],
                    colors=[x[2] for x in non_zero], autopct='%1.1f%%', startangle=90)
        ax2.set_title('FTMO Outcome Distribution')

        plt.tight_layout()
        plt.show()

In [ ]:
# UNIVERSAL EXPORT

import json, datetime, uuid

if len(results_df) == 0:
    print('No results to export.')
else:
    STRATEGY_NAME = 'VWAP_Bounce'
    PARAM_COLS = ['vwap_period', 'pullback_pct', 'slope_lookback']

    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    best_params = {col: (int(best[col]) if col != 'pullback_pct' else float(best[col])) for col in PARAM_COLS}
    param_str = ', '.join(f'{k}={v}' for k, v in best_params.items())

    # Rebuild IS / OOS portfolios for metrics
    ent_is, ext_is = generate_vwap_bounce_signals(train_close, train_high, train_low, train_volume, **best_params)
    pf_is = vbt.Portfolio.from_signals(train_close, entries=ent_is, exits=ext_is,
                                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')

    ent_oos, ext_oos = generate_vwap_bounce_signals(val_close, val_high, val_low, val_volume, **best_params)
    pf_oos = vbt.Portfolio.from_signals(val_close, entries=ent_oos, exits=ext_oos,
                                         init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D')

    M = m_full  # from cell 12
    m_is_dict = extract_full_metrics(pf_is, max((train_close.index[-1] - train_close.index[0]).days / 365.25, 1e-9))
    m_oos_dict = extract_full_metrics(pf_oos, max((val_close.index[-1] - val_close.index[0]).days / 365.25, 1e-9))

    RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

    # Build export JSON
    export_json = {
        'metadata': {
            'run_id': RUN_ID,
            'strategy_name': STRATEGY_NAME,
            'strategy_family': 'VWAP',
            'ticker': TICKER,
            'start_date': str(close.index[0].date()),
            'end_date': str(close.index[-1].date()),
            'total_bars': len(close),
            'train_ratio': TRAIN_RATIO,
            'init_cash': INIT_CASH,
            'fees_pct': FEES,
            'slippage_pct': SLIPPAGE,
            'frequency': 'D',
            'notes': f'Brian Shannon VWAP Bounce. {param_str}'
        },
        'best_params': best_params,
        'metrics_full_sample': {k: v for k, v in M.items() if not k.startswith('is_') and not k.startswith('oos_') and not k.startswith('bh_')},
        'metrics_in_sample': {'sharpe': m_is_dict['sharpe_ratio'], 'return': m_is_dict['total_return'],
                              'dd': m_is_dict['max_drawdown'], 'trades': m_is_dict['total_trades']},
        'metrics_out_of_sample': {'sharpe': m_oos_dict['sharpe_ratio'], 'return': m_oos_dict['total_return'],
                                  'dd': m_oos_dict['max_drawdown'], 'trades': m_oos_dict['total_trades']},
        'metrics_buy_hold': {'sharpe': m_bh['sharpe_ratio'], 'return': m_bh['total_return'], 'dd': m_bh['max_drawdown']},
        'monte_carlo_ftmo': {
            'pass_rate': pass_rate if 'pass_rate' in dir() else 0.0,
            'verdict': verdict if 'verdict' in dir() else 'N/A'
        },
        'grid_search_summary': {
            'top5': results_df.nlargest(5, 'sharpe_ratio')[PARAM_COLS + ['sharpe_ratio', 'total_return', 'max_drawdown']].to_dict('records')
        }
    }

    # Save paths
    try:
        base_dir = f'/content/drive/MyDrive/strategy_exports/{STRATEGY_NAME}/{TICKER}/latest'
        os.makedirs(base_dir, exist_ok=True)
    except:
        base_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'exports', STRATEGY_NAME, TICKER, 'latest')
        os.makedirs(base_dir, exist_ok=True)

    # Save JSON
    with open(os.path.join(base_dir, 'summary.json'), 'w') as f:
        json.dump(export_json, f, indent=2, default=str)
    print(f'Summary saved to: {os.path.join(base_dir, "summary.json")}')

    # Save trades CSV
    try:
        trades_obj = pf_full.trades
        tr = np.asarray(trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns).ravel()
        pnl_arr = np.asarray(trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl).ravel()
        pd.DataFrame({'trade_num': range(1, len(tr)+1), 'return_pct': tr*100, 'pnl_usd': pnl_arr,
                      'cumulative_pnl': np.cumsum(pnl_arr), 'is_winner': tr > 0
        }).to_csv(os.path.join(base_dir, 'trades.csv'), index=False)
        print(f'Trades saved')
    except Exception as e:
        print(f'Could not save trades: {e}')

    # Save daily returns CSV
    daily_ret_series = pf_full.daily_returns()
    pd.DataFrame({'date': close.index.strftime('%Y-%m-%d'), 'strategy_return': daily_ret_series.values,
                  'close': close.values, 'portfolio_value': pf_full.value().values
    }).to_csv(os.path.join(base_dir, 'daily_returns.csv'), index=False)
    print(f'Daily returns saved')

    # Save grid results CSV
    results_df.to_csv(os.path.join(base_dir, 'grid_results.csv'), index=False)
    print(f'Grid results saved')

    # Google Sheets logging
    try:
        from lib.sheets_logger import SheetsLogger
        _logger = SheetsLogger()
        _logger.log_result(export_json)
    except Exception as e:
        print(f'Google Sheets logging skipped: {e}')

    print('\nExport complete!')